# OpenAI Agents SDK: Documentation RAG Agent

This notebook recreates the same documentation RAG agent using the **OpenAI Agents SDK**.

The SDK provides decorators for defining tools and a `Runner` that handles the agent loop: model calls, tool execution, tool-result handling, and final output.

## Install dependency

```bash
uv add openai-agents
```

### Short production reminder

For production, replace print-based callbacks with structured logs or tracing events. Include request/run IDs, tool name, latency, status, and redacted argument summaries. Do not log sensitive content or full tool arguments by default. Keep callbacks observational; enforce authorization and validation inside the tool execution boundary.


In [ ]:
from search_tools import create_search_tools
from common import AGENT_INSTRUCTIONS

search_tools = create_search_tools()

## Define tools

The SDK uses `@function_tool`. Each function needs a docstring because the SDK uses it to describe the tool to the model.

In [ ]:
from agents import function_tool

@function_tool
def search(query: str) -> list:
    """Search the documentation index for results matching a query."""
    return search_tools.search(query)

@function_tool
def get_file(filename: str) -> str:
    """Retrieve a file's full contents by filename."""
    return search_tools.get_file(filename)

tools = [search, get_file]

## Create the agent

The `Agent` object takes a name, model, instructions, and tools.

In [ ]:
from agents import Agent

search_agent = Agent(
    name="search_agent",
    model="gpt-4o-mini",
    instructions=AGENT_INSTRUCTIONS,
    tools=tools,
)

## Run the agent

`Runner.run()` manages the agentic loop and returns a `RunResult`. The final answer is in `result.final_output`.

In [ ]:
from agents import Runner

query = "how do I use evidently to monitor my machine learning models?"

result = await Runner.run(search_agent, input=query)

print(result.final_output)

## Inspect new items

`result.new_items` contains the messages and tool activity generated during this run.

- `message_output_item`: model text output
- `tool_call_item`: model request to call a tool
- `tool_call_output_item`: result returned by the tool

In [ ]:
def print_agent_items(items):
    for item in items:
        item_type = item.type

        if item_type == "message_output_item":
            for content in item.raw_item.content:
                if hasattr(content, "text"):
                    print(f"Assistant: {content.text[:300]}")

        elif item_type == "tool_call_item":
            tool_call = item.raw_item
            print(f"Tool call: {tool_call.name}({tool_call.arguments})")

        elif item_type == "tool_call_output_item":
            print("Tool call output")

        print()


print_agent_items(result.new_items)

## Follow-up conversation

Use `result.to_input_list()` to convert the prior run into an input list, then append the next user message.

In [ ]:
follow_up = "show me the code"

messages = result.to_input_list()
messages.append({"role": "user", "content": follow_up})

result2 = await Runner.run(search_agent, input=messages)

print(result2.final_output)

In [ ]:
print_agent_items(result2.new_items)

## Alternative tool wrapping

Instead of redefining functions with `@function_tool`, wrap existing methods directly. This is useful when the original methods already have useful docstrings.

In [ ]:
tools_v2 = [
    function_tool(search_tools.search),
    function_tool(search_tools.get_file),
]

search_agent_v2 = Agent(
    name="search_agent",
    model="gpt-4o-mini",
    instructions=AGENT_INSTRUCTIONS,
    tools=tools_v2,
)

result3 = await Runner.run(search_agent_v2, input=query)

print(result3.final_output)

## Sessions for multi-turn conversations

`SQLiteSession` automatically stores and retrieves conversation history. You no longer need to manage message lists manually.

By default, it stores history in memory. To persist across restarts, provide a database path.

In [ ]:
from agents import SQLiteSession

# In-memory session
session = SQLiteSession("docs-conversation")

result_session = await Runner.run(
    search_agent,
    "how do I use evidently to monitor my machine learning models?",
    session=session,
)

print(result_session.final_output[:300] + "...")

In [ ]:
# Same session automatically includes prior history
result_session2 = await Runner.run(
    search_agent,
    "show me the code",
    session=session,
)

print(result_session2.final_output[:300] + "...")

In [ ]:
# Persistent session across notebook/server restarts
persistent_session = SQLiteSession(
    "docs-conversation",
    "conversations.db",
)

## Production-style observability

Use structured logs or tracing in production. The function below records tool calls after a run finishes; it does not execute tools or make authorization decisions.

In [ ]:
import time
import logging
import uuid

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s %(message)s",
)

logger = logging.getLogger("openai_agents_docs")


def summarize_args(args, limit: int = 120) -> str:
    text = str(args)
    return text if len(text) <= limit else text[:limit] + "..."


def log_agent_run(result, request_id: str) -> None:
    """Observational post-run logging for an OpenAI Agents SDK result."""
    for item in result.new_items:
        if item.type != "tool_call_item":
            continue

        tool_call = item.raw_item
        logger.info(
            "agent_tool_call",
            extra={
                "event": "tool_call",
                "request_id": request_id,
                "agent_name": search_agent.name,
                "tool_name": tool_call.name,
                "args_summary": summarize_args(tool_call.arguments),
                "status": "requested",
            },
        )


request_id = str(uuid.uuid4())
started = time.perf_counter()

logged_result = await Runner.run(search_agent, input=query)

logger.info(
    "agent_run_completed",
    extra={
        "event": "agent_run_completed",
        "request_id": request_id,
        "agent_name": search_agent.name,
        "latency_ms": round((time.perf_counter() - started) * 1000, 2),
        "status": "success",
    },
)

log_agent_run(logged_result, request_id)

print(logged_result.final_output)

## Client production checklist

- Authenticate users and isolate data by tenant.
- Give tools explicit permissions.
- Validate inputs and authorize actions inside tool functions.
- Use structured logs or traces with request IDs.
- Add timeouts, retries, rate limits, and cost controls.
- Avoid logging raw user content or complete tool arguments.
- Test normal, unauthorized, invalid-input, and upstream-failure paths.